In [0]:
from pyspark.sql.functions import col, when

In [0]:
# Create dbutils widget for config_path with a default value
dbutils.widgets.text("config_path", "./configs/dev.yaml")

# Retrieve the widget value (either passed from job or uses default)
config_path = dbutils.widgets.get("config_path")
print(f"Config path in use: {config_path}")


In [0]:
import yaml

# Load config (adjust path as needed)
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

RAW_CSV_PATH = config['raw_csv_path']
SCHEMA = config['schema']
BRONZE_TABLE = config['bronze_table']
SILVER_TABLE = config['silver_table']

In [0]:
TRANSFORM_CSV_TO_BRONZE = True
TRANSFORM_BRONZE_TO_SILVER = True

# Transform csv file from stage to bronze layer

In [0]:
if TRANSFORM_CSV_TO_BRONZE:
    # Read CSV into Spark DataFrame
    df = spark.read.csv(RAW_CSV_PATH, header=True, inferSchema=True)
    # Save as a managed Delta table (overwrite if re-running for idempotency)
    df.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA}.{BRONZE_TABLE}")

# Transform from bronze to silver layer

In [0]:
if TRANSFORM_BRONZE_TO_SILVER:
    # Read from managed table as DataFrame
    bronze_df = spark.table(f"{SCHEMA}.{BRONZE_TABLE}")
    
    # Example simple transformation: select useful columns and cast TotalCharges to float
    df_transformed = bronze_df.withColumn(
        "TotalCharges",
        when(col("TotalCharges") == " ", None).otherwise(col("TotalCharges").cast("float"))
    ).select("customerID", "gender", "SeniorCitizen", "tenure", "TotalCharges", "Churn")

    # Write to new table for downstream steps
    df_transformed.write.format("delta").mode("overwrite").saveAsTable(f"{SCHEMA}.{SILVER_TABLE}")
